In [ ]:
%pip install torch torchvision pillow transformers fastapi

- Standard: ZhengPeng7/BiRefNet (High accuracy)
- Lite: ZhengPeng7/BiRefNet_lite (Faster, smaller)
- Portrait: ZhengPeng7/BiRefNet_portrait (Specialized for people)

In [9]:
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

# 1. Load the model (Automatically downloads from Hugging Face)
model_name = "ZhengPeng7/BiRefNet_lite" 
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForImageSegmentation.from_pretrained(model_name, trust_remote_code=True)
model.to(device)
model.eval()

# 2. Prepare the Image
def bg_image_remover(image_path):
    input_image = Image.open(image_path).convert("RGB")
    
    # Standard BiRefNet normalization
    transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    input_tensor = transform(input_image).unsqueeze(0).to(device)
    
    # Convert input to match model's dtype
    input_tensor = input_tensor.to(next(model.parameters()).dtype)
    
    # 3. Inference
    with torch.no_grad():
        preds = model(input_tensor)[-1].sigmoid().cpu()
        pred = preds[0].squeeze()
    
    # 4. Create Mask and Apply
    mask = transforms.ToPILImage()(pred)
    mask = mask.resize(input_image.size)
    
    input_image.putalpha(mask) # Adds transparency based on the mask
    return input_image


Loading weights: 100%|██████████| 586/586 [00:00<00:00, 2209.36it/s]


In [ ]:
# Run it
result = bg_image_remover("8865880.jpg")
result.save("output_no_bg.png")

# WaterMark Remover

In [10]:
%pip install onnxruntime Pillow numpy huggingface_hub

Note: you may need to restart the kernel to use updated packages.
